In [1]:
# CELL 1: Environment, paths, and load Stage 2 outputs
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import util

EMB_DIR = Path("../data/embeddings")
OUTPUT_DIR = Path("../data/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample = pd.read_csv(EMB_DIR / "cicids_sample.csv", index_col="sample_id", low_memory=False)
embeddings = np.load(EMB_DIR / "cicids_embeddings.npy")
assert len(sample) == len(embeddings), "Embedding/CSV length mismatch"
print(f"Loaded {len(sample)} samples | Shape: {embeddings.shape}")

Loaded 10684 samples | Shape: (10684, 384)


In [2]:
# CELL 2: Community detection — direct on embedding space (course method)
# The professor is explicit: community detection operates on the embedding similarity
# matrix directly. No graph is built at this stage. util.community_detection does
# exactly this — greedy cosine-threshold grouping.
print("Running community detection on embedding space...")
communities = util.community_detection(
    embeddings,
    min_community_size=3,
    threshold=0.75,
)
print(f"Found {len(communities)} semantic communities")

Running community detection on embedding space...
Found 19 semantic communities


In [3]:
# CELL 3: Map communities to DataFrame
community_assignments = np.full(len(sample), -1, dtype=int)
for idx, comm_indices in enumerate(communities):
    for sample_idx in comm_indices:
        community_assignments[sample_idx] = idx

valid_mask = community_assignments != -1
community_df = sample[valid_mask].copy()
community_df["community_id"] = community_assignments[valid_mask]
community_df.to_csv(OUTPUT_DIR / "community_assignments.csv", index=False)
print(f"Assigned {len(community_df)} alerts to {community_df['community_id'].nunique()} communities")

# Quick sanity check: what does each community look like?
print("\nCommunity composition preview:")
for cid in sorted(community_df['community_id'].unique())[:5]:
    group = community_df[community_df['community_id'] == cid]
    dominant_label = group['Label'].mode().iloc[0]
    dominant_tactic = group['attck_tactic'].mode().iloc[0]
    print(f"  Community {cid}: {len(group)} alerts | {dominant_label} | {dominant_tactic}")

Assigned 10684 alerts to 19 communities

Community composition preview:
  Community 0: 8665 alerts | PortScan | Impact
  Community 1: 1013 alerts | FTP Patator | Credential Access
  Community 2: 329 alerts | BENIGN | Benign
  Community 3: 245 alerts | BENIGN | Benign
  Community 4: 148 alerts | FTP Patator | Credential Access


In [5]:
# CELL 4: LLM initialisation
# Using Qwen2.5-8B-Instruct (Q4_K_M) — significantly better instruction-following
# than the 3B for structured extraction tasks. On 4GB VRAM, set n_gpu_layers to
# offload most layers (~20-24 of 32) to GPU and let the rest run on CPU RAM.
# This is slower but fully functional on the RTX 3050 Ti.
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=20,      # Offload 20 layers to GPU; rest on CPU — adjust based on available VRAM
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=42,
)
print(f"LLM loaded: {MODEL_PATH}")

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded: ../models/qwen2.5-3b-instruct-q4_k_m.gguf


In [6]:
# CELL 5: Grammar schema — OPEN subject and target fields
#
# THE CORE FIX: The previous version used enums for subject AND target, which
# meant the LLM was picking from a hardcoded menu rather than reading the alert
# content. Every triple came out as "network_flow -> X" with no actual extracted
# entities. The knowledge graph this produced had 12 nodes and no semantic content.
#
# The correct approach (per the professor's description of his OSINT pipeline):
# - subject and target are FREE strings — the LLM names what it actually sees
# - relation is constrained to a small enum for consistency across communities
# - The grammar enforces JSON structure, NOT vocabulary
#
# This means the graph will have real, discovered entities as nodes:
# "ssh_brute_force_client", "port_22_authentication", "http_flood_source", etc.

TRIPLE_SCHEMA = {
    "type": "object",
    "properties": {
        "triples": {
            "type": "array",
            "minItems": 4,
            "maxItems": 4,
            "items": {
                "type": "object",
                "properties": {
                    "subject": {
                        "type": "string"
                        # FREE — LLM extracts the actual acting entity from the text
                        # e.g. "ssh_scanner", "http_flood_source", "ftp_brute_force_client"
                    },
                    "relation": {
                        "type": "string",
                        "enum": [
                            "targets",
                            "executes",
                            "communicates_with",
                            "generates",
                            "exploits",
                            "authenticates_to",
                            "scans",
                            "floods"
                        ]
                        # Constrained: keeps relations consistent across communities
                        # so the graph edges are comparable and meaningful
                    },
                    "target": {
                        "type": "string"
                        # FREE — LLM extracts the actual target entity from the text
                        # e.g. "port_22_service", "http_web_server", "dns_resolver"
                    }
                },
                "required": ["subject", "relation", "target"]
            }
        }
    },
    "required": ["triples"]
}

grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print("Grammar initialized — subject/target are open strings, relation is constrained enum")

Grammar initialized — subject/target are open strings, relation is constrained enum


In [7]:
# CELL 6: Triple extraction function
#
# Prompt design:
# - Tell the model it is a cybersecurity analyst
# - Give it the actual alert texts to read (not just metadata)
# - Instruct it to NAME the specific actors and targets it observes
# - Provide a concrete example so it understands the expected output format
# - Use temperature=0 + seed=42 for reproducibility

def normalise_entity(text: str) -> str:
    """Lowercase, strip, replace spaces/special chars with underscores.
    Keeps entity names consistent for graph node matching."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:80]  # cap length to avoid runaway entity names


def extract_triples(alert_texts: list, community_id: int) -> list:
    """
    Given a list of alert_text strings from one community, ask the LLM
    to read them and extract 4 semantic triples describing what is happening.
    Returns a list of 4 validated triple dicts.
    """
    block = "\n".join([f"- {t}" for t in alert_texts])

    prompt = f"""[INST] You are a cybersecurity analyst. Read these network alerts and extract exactly 4 semantic triples.

For each triple:
- subject: name the specific actor or source you observe (e.g. "ssh_scanner", "http_flood_source", "ftp_client")
- relation: choose from [targets, executes, communicates_with, generates, exploits, authenticates_to, scans, floods]
- target: name the specific service, port, or system being acted on (e.g. "ssh_port_22", "http_web_server", "dns_resolver")

Name what you actually observe in the text. Do not use generic placeholders.

Example output:
{{"triples": [
  {{"subject": "ssh_brute_force_client", "relation": "targets", "target": "ssh_port_22_service"}},
  {{"subject": "ssh_brute_force_client", "relation": "generates", "target": "high_volume_short_connections"}},
  {{"subject": "attacker", "relation": "authenticates_to", "target": "ssh_authentication_endpoint"}},
  {{"subject": "credential_guessing_process", "relation": "executes", "target": "password_spray_attack"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt,
        max_tokens=512,
        temperature=0,
        seed=42,
        grammar=grammar,
        repeat_penalty=1.1,
        stop=["[/INST]"]
    )
    raw = out["choices"][0]["text"].strip()

    try:
        parsed = json.loads(raw)
        triples = parsed.get("triples", [])
    except Exception as e:
        print(f"  [Community {community_id}] JSON parse failed: {e}")
        triples = []

    # Validate structure and normalise entity names
    validated = []
    valid_relations = {"targets", "executes", "communicates_with", "generates",
                       "exploits", "authenticates_to", "scans", "floods"}

    for t in triples:
        subj = normalise_entity(str(t.get("subject", "")))
        rel  = str(t.get("relation", "")).lower().strip()
        tgt  = normalise_entity(str(t.get("target", "")))

        # Skip triples with empty fields
        if not subj or not tgt:
            continue
        # Fall back to 'targets' if relation not in enum (grammar should prevent this)
        if rel not in valid_relations:
            rel = "targets"

        validated.append({"subject": subj, "relation": rel, "target": tgt})

    # Pad to 4 if LLM returned fewer (fallback only)
    fallback = {"subject": f"community_{community_id}_actor",
                "relation": "targets",
                "target": f"community_{community_id}_target"}
    validated = (validated + [fallback] * 4)[:4]
    return validated

In [8]:
# CELL 7: Warmup and extraction loop
print("Warming up LLM...")
_ = llm("[INST] Test. [/INST]", max_tokens=5, temperature=0, seed=42)
print("Warmup complete. Starting extraction...\n")

community_triples = {}
for cid, group in community_df.groupby("community_id"):
    # Take up to 6 representative alerts per community
    texts = group["alert_text"].dropna().head(6).tolist()
    triples = extract_triples(texts, int(cid))
    community_triples[str(int(cid))] = triples

    # Live inspection
    dominant_label = group['Label'].mode().iloc[0]
    print(f"Community {cid} [{dominant_label}]:")
    for t in triples:
        print(f"  {t['subject']} --[{t['relation']}]--> {t['target']}")
    print()

print(f"Extraction complete: {len(community_triples)} communities processed")

Warming up LLM...
Warmup complete. Starting extraction...

Community 0 [PortScan]:
  ftp_client --[targets]--> ftp_port_21_service
  ftp_client --[scans]--> http_web_server
  http_flood_source --[floods]--> http_web_server
  http_flood_source --[targets]--> http_port_80_service

Community 1 [FTP Patator]:
  ssh_brute_force_client --[targets]--> ssh_port_22_service
  http_flood_source --[floods]--> http_web_server
  ftp_client --[communicates_with]--> ftp_server
  credential_guessing_process --[executes]--> password_spray_attack

Community 2 [BENIGN]:
  dns_client --[targets]--> dns_resolver
  ftp_client --[targets]--> ftp_server
  dns_client --[generates]--> multiple_dns_queries
  ftp_client --[executes]--> ftp_operation

Community 3 [BENIGN]:
  dns_resolver --[targets]--> http_web_server
  dns_resolver --[targets]--> ssh_port_22_service
  dns_resolver --[targets]--> ftp_client
  dns_resolver --[scans]--> http_web_server

Community 4 [FTP Patator]:
  ftp_client --[targets]--> ftp_port_

In [9]:
# CELL 8: Validate and save outputs

# Count unique entities across all triples — a key quality signal.
# With the old enum approach this number was ~12 (just the enum values).
# With open extraction it should be much higher, reflecting real diversity.
all_subjects = set()
all_targets  = set()
valid_count  = 0
total_count  = 0

for triples in community_triples.values():
    for t in triples:
        total_count += 1
        if all(k in t and t[k] for k in ["subject", "relation", "target"]):
            valid_count += 1
            all_subjects.add(t["subject"])
            all_targets.add(t["target"])

triple_metrics = {
    "n_communities": len(community_triples),
    "total_triples": total_count,
    "valid_triples": valid_count,
    "valid_ratio": round(valid_count / max(total_count, 1), 4),
    "unique_subjects": len(all_subjects),
    "unique_targets": len(all_targets),
    "unique_entities_total": len(all_subjects | all_targets),
    # NOTE: unique_entities_total >> n_communities means the LLM
    # is actually discovering diverse entities, not just repeating enum values
}

with open(OUTPUT_DIR / "triple_metrics.json", "w") as f:
    json.dump(triple_metrics, f, indent=2)

with open(OUTPUT_DIR / "community_triples.json", "w", encoding="utf-8") as f:
    json.dump(community_triples, f, indent=2)

print("=== TRIPLE EXTRACTION METRICS ===")
for k, v in triple_metrics.items():
    print(f"  {k}: {v}")

print(f"\nAll unique subjects discovered: {sorted(all_subjects)}")
print(f"\nAll unique targets discovered:  {sorted(all_targets)}")
print("\nStage 3 complete. Outputs saved to", OUTPUT_DIR)

=== TRIPLE EXTRACTION METRICS ===
  n_communities: 19
  total_triples: 76
  valid_triples: 76
  valid_ratio: 1.0
  unique_subjects: 20
  unique_targets: 21
  unique_entities_total: 38

All unique subjects discovered: ['attacker', 'credential_guessing_process', 'dns_client', 'dns_resolver', 'dns_responder', 'ftp_client', 'high_volume_traffic_pattern', 'http_flood_source', 'https_client', 'rdp_client', 'ssh_brute_force_client', 'ssh_client', 'unknown_port_10002', 'unknown_port_1098', 'unknown_port_1723', 'unknown_port_3289', 'unknown_port_52822', 'unknown_port_5353', 'unknown_port_8180', 'very_short_connection_behavior']

All unique targets discovered:  ['credential_guessing_attack', 'credential_guessing_process', 'dns_resolver', 'ftp_client', 'ftp_login_attempt', 'ftp_operation', 'ftp_port_21_service', 'ftp_server', 'ftp_server_vulnerability', 'high_volume_short_connections', 'http_authentication_endpoint', 'http_port_80_service', 'http_web_server', 'https_port_443_service', 'https_web_